You are a data analyst working for the Australian Renewable Energy Agency (ARENA). 
ARENA is interested in understanding the factors influencing solar energy adoption across different states and territories in Australia. 

After your preliminary report, ARENA wants you to **focus on the Melbourne residential market**. 
They want to know **whether there are distinct solar adoption patterns, which may require different policies and incentives**.  

Your task is to gather, analyse, and report on solar installation data and combine in with sales data for the Melbourne market, to help ARENA make informed decisions about future solar energy projects and policies. 

ARENA has provided you with data about solar unit adoption and energy generation for all Australian suburbs and has sourced detailed housing sales reports for Melbourne. ARENA expects you to obtain additional data sources to complete the analysis as required. 

In [4]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import numpy as np

In [20]:
# convert to state from postcode and add a state column
def get_state_from_postcode(postcode):
    try:
        code = int(postcode)
    except:
        return "Invalid"

    if (200<=code<=299 or 2600<=code<=2639 or 2900<=code<=2920):
        return "ACT"
    elif(800<=code<=999):
        return "NT"
    elif(1000<=code<=1999 or 2000<=code<=2599 or 2640<=code<=2914):
        return "NSW"
    elif(3000<=code<=3999 or 8000<=code<=8999):
        return "VIC"
    elif(4000<=code<=4999 or 9000<=code<=9999):
        return "QLD"
    elif(5000<=code<=5999):
        return "SA"
    elif(6000<=code<=6999):
        return "WA"
    elif(7000<=code<=7999):
        return "TAS"
    else:
        return "Unknown"

In [21]:
# post code
df_postcode = pd.read_csv('australian_postcodes.csv')
df_vic_postcode = df_postcode[df_postcode['state'] == 'VIC']
df_vic_postcode.head()

,id,postcode,locality,state,long,lat,dc,type,status,sa3,...,altitude,chargezone,phn_code,phn_name,lgaregion,lgacode,electorate,electoraterating,sed_code,sed_name
6202,4746,3000,MELBOURNE,VIC,144.982585,-37.814437,CITY DELIVERY CENTRE,Delivery Area,Updated 17-Mar-2024 AUSPOST,20604.0,...,27.332188,V1,PHN201,North Western Melbourne,Melbourne,24600.0,Melbourne,Inner Metropolitan,24703.0,Melbourne (Northern Metropolitan)
6203,4747,3001,MELBOURNE,VIC,144.982585,-37.814437,CITY MAIL PROCESSING CENTRE,Post Office Boxes,Updated 17-Mar-2024 AUSPOST,20605.0,...,27.332188,V1,PHN203,NaN,Melbourne,24600.0,Maribyrnong,NaN,NaN,NaN
6204,4748,3002,EAST MELBOURNE,VIC,144.982585,-37.814437,CITY DELIVERY CENTRE,Delivery Area,Updated 17-Mar-2024 AUSPOST,20604.0,...,27.332188,V1,PHN201,North Western Melbourne,Melbourne,24600.0,Melbourne,Inner Metropolitan,24703.0,Melbourne (Northern Metropolitan)
6205,4749,3003,WEST MELBOURNE,VIC,144.949592,-37.810871,CITY DELIVERY CENTRE,Delivery Area,Updated 6-Feb-2020,20604.0,...,15.316063,V1,PHN201,North Western Melbourne,Melbourne,24600.0,Melbourne,Inner Metropolitan,24703.0,Melbourne (Northern Metropolitan)
6206,4750,3004,MELBOURNE,VIC,144.982585,-37.814437,CITY DELIVERY CENTRE,Delivery Area,Updated 17-Mar-2024 AUSPOST,20605.0,...,27.332188,V1,PHN203,South Eastern Melbourne,Melbourne,24600.0,Macnamara,Inner Metropolitan,26906.0,Prahran (Southern Metropolitan)


In [22]:
df_mel = df_vic_postcode.iloc[:, 1]
df_mel

6202     3000
6203     3001
6204     3002
6205     3003
6206     3004
         ... 
18529    8438
18530    8511
18531    8785
18532    8785
18544    9999
Name: postcode, Length: 3541, dtype: int64

In [23]:
#capacity
df_capacity = pd.read_csv('sgu-solar-capacity-2011-to-present.csv')
df_capacity.head()

,Small Unit Installation Postcode,Jan 2011 - SGU Rated Output In kW,Feb 2011 - SGU Rated Output In kW,Mar 2011 - SGU Rated Output In kW,Apr 2011 - SGU Rated Output In kW,May 2011 - SGU Rated Output In kW,Jun 2011 - SGU Rated Output In kW,Jul 2011 - SGU Rated Output In kW,Aug 2011 - SGU Rated Output In kW,Sep 2011 - SGU Rated Output In kW,...,Jul 2023 - SGU Rated Output In kW,Aug 2023 - SGU Rated Output In kW,Sep 2023 - SGU Rated Output In kW,Oct 2023 - SGU Rated Output In kW,Nov 2023 - SGU Rated Output In kW,Dec 2023 - SGU Rated Output In kW,Jan 2024 - SGU Rated Output In kW,Feb 2024 - SGU Rated Output In kW,Mar 2024 - SGU Rated Output In kW,Apr 2024 - SGU Rated Output In kW
0,0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.0,...,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0
1,200,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.0,...,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0
2,800,0.0,0.0,0.0,0.0,2.22,0.0,3.04,0.0,0.0,...,39.96,55.15,0.0,0.0,62.25,0.0,0.0,0.0,0.0,0.0
3,801,0.0,0.0,0.0,0.0,2.02,0.0,0.00,0.0,0.0,...,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0
4,803,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.0,...,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0


In [24]:
df_capacity['State'] = df_capacity['Small Unit Installation Postcode'].apply(get_state_from_postcode)
state_col = df_capacity.pop('State')
df_capacity.insert(1, 'State', state_col)
df_capacity_melbourne = df_capacity[df_capacity['State'] == 'VIC']
df_capacity_melbourne.head()

C:\Users\aynss\AppData\Local\Temp\ipykernel_26568\2742442017.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_capacity['State'] = df_capacity['Small Unit Installation Postcode'].apply(get_state_from_postcode)
C:\Users\aynss\AppData\Local\Temp\ipykernel_26568\2742442017.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_capacity.insert(1, 'State', state_col)


,Small Unit Installation Postcode,State,Jan 2011 - SGU Rated Output In kW,Feb 2011 - SGU Rated Output In kW,Mar 2011 - SGU Rated Output In kW,Apr 2011 - SGU Rated Output In kW,May 2011 - SGU Rated Output In kW,Jun 2011 - SGU Rated Output In kW,Jul 2011 - SGU Rated Output In kW,Aug 2011 - SGU Rated Output In kW,...,Jul 2023 - SGU Rated Output In kW,Aug 2023 - SGU Rated Output In kW,Sep 2023 - SGU Rated Output In kW,Oct 2023 - SGU Rated Output In kW,Nov 2023 - SGU Rated Output In kW,Dec 2023 - SGU Rated Output In kW,Jan 2024 - SGU Rated Output In kW,Feb 2024 - SGU Rated Output In kW,Mar 2024 - SGU Rated Output In kW,Apr 2024 - SGU Rated Output In kW
736,3000,VIC,0.0,0.0,0.00,0.0,1.52,5.27,0.00,0.0,...,84.41,86.86,65.00,19.8,30.25,96.36,0.0,0.0,0.0,0.0
737,3002,VIC,0.0,0.0,0.00,0.0,1.90,3.00,3.04,0.0,...,7.65,19.55,0.00,0.0,6.16,0.00,0.0,4.8,0.0,0.0
738,3003,VIC,0.0,0.0,1.48,0.0,2.09,9.54,0.00,0.0,...,64.40,58.10,0.00,0.0,6.72,38.72,0.0,0.0,0.0,0.0
739,3004,VIC,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.00,0.00,0.0,29.88,17.10,0.0,0.0,0.0,0.0
740,3006,VIC,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.00,15.54,0.0,0.00,0.00,0.0,39.0,0.0,0.0


In [29]:
capacity_cols = df_capacity_melbourne.columns[2:]
# Total capacity (kW)
df_capacity_melbourne['Total_Capacity_kW'] = df_capacity_melbourne[capacity_cols].sum(axis=1)
df_capacity_melbourne.head()

,Small Unit Installation Postcode,State,Jan 2011 - SGU Rated Output In kW,Feb 2011 - SGU Rated Output In kW,Mar 2011 - SGU Rated Output In kW,Apr 2011 - SGU Rated Output In kW,May 2011 - SGU Rated Output In kW,Jun 2011 - SGU Rated Output In kW,Jul 2011 - SGU Rated Output In kW,Aug 2011 - SGU Rated Output In kW,...,Aug 2023 - SGU Rated Output In kW,Sep 2023 - SGU Rated Output In kW,Oct 2023 - SGU Rated Output In kW,Nov 2023 - SGU Rated Output In kW,Dec 2023 - SGU Rated Output In kW,Jan 2024 - SGU Rated Output In kW,Feb 2024 - SGU Rated Output In kW,Mar 2024 - SGU Rated Output In kW,Apr 2024 - SGU Rated Output In kW,Total_Capacity_kW
736,3000,VIC,0.0,0.0,0.00,0.0,1.52,5.27,0.00,0.0,...,86.86,65.00,19.8,30.25,96.36,0.0,0.0,0.0,0.0,5158.312
737,3002,VIC,0.0,0.0,0.00,0.0,1.90,3.00,3.04,0.0,...,19.55,0.00,0.0,6.16,0.00,0.0,4.8,0.0,0.0,1521.170
738,3003,VIC,0.0,0.0,1.48,0.0,2.09,9.54,0.00,0.0,...,58.10,0.00,0.0,6.72,38.72,0.0,0.0,0.0,0.0,3659.060
739,3004,VIC,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.00,0.0,29.88,17.10,0.0,0.0,0.0,0.0,884.880
740,3006,VIC,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,15.54,0.0,0.00,0.00,0.0,39.0,0.0,0.0,2643.510


In [31]:
# Recent 1 year (May 2023 ~ Apr 2024)
recent_capacity_cols = [c for c in capacity_cols if '2023' in c or '2024' in c]
df_capacity_melbourne['Recent_Capacity_kW'] = df_capacity_melbourne[recent_capacity_cols].sum(axis=1)
df_capacity_melbourne.head()

,Small Unit Installation Postcode,State,Jan 2011 - SGU Rated Output In kW,Feb 2011 - SGU Rated Output In kW,Mar 2011 - SGU Rated Output In kW,Apr 2011 - SGU Rated Output In kW,May 2011 - SGU Rated Output In kW,Jun 2011 - SGU Rated Output In kW,Jul 2011 - SGU Rated Output In kW,Aug 2011 - SGU Rated Output In kW,...,Sep 2023 - SGU Rated Output In kW,Oct 2023 - SGU Rated Output In kW,Nov 2023 - SGU Rated Output In kW,Dec 2023 - SGU Rated Output In kW,Jan 2024 - SGU Rated Output In kW,Feb 2024 - SGU Rated Output In kW,Mar 2024 - SGU Rated Output In kW,Apr 2024 - SGU Rated Output In kW,Total_Capacity_kW,Recent_Capacity_kW
736,3000,VIC,0.0,0.0,0.00,0.0,1.52,5.27,0.00,0.0,...,65.00,19.8,30.25,96.36,0.0,0.0,0.0,0.0,5158.312,716.670
737,3002,VIC,0.0,0.0,0.00,0.0,1.90,3.00,3.04,0.0,...,0.00,0.0,6.16,0.00,0.0,4.8,0.0,0.0,1521.170,82.910
738,3003,VIC,0.0,0.0,1.48,0.0,2.09,9.54,0.00,0.0,...,0.00,0.0,6.72,38.72,0.0,0.0,0.0,0.0,3659.060,285.975
739,3004,VIC,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,29.88,17.10,0.0,0.0,0.0,0.0,884.880,106.880
740,3006,VIC,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,...,15.54,0.0,0.00,0.00,0.0,39.0,0.0,0.0,2643.510,176.840


In [33]:
# 必要な列だけ残す
df_cap_features = df_capacity_melbourne[['Small Unit Installation Postcode', 'Total_Capacity_kW', 'Recent_Capacity_kW']].copy()
df_cap_features.rename(columns={'Small Unit Installation Postcode': 'Postcode'}, inplace=True)
df_cap_features.head()

,Postcode,Total_Capacity_kW,Recent_Capacity_kW
736,3000,5158.312,716.670
737,3002,1521.170,82.910
738,3003,3659.060,285.975
739,3004,884.880,106.880
740,3006,2643.510,176.840


In [26]:
#installations
df_installations = pd.read_csv('sgu-solar-installations-2011-to-present.csv')
df_installations.head()

,Small Unit Installation Postcode,Jan 2011 - Installations Quantity,Feb 2011 - Installations Quantity,Mar 2011 - Installations Quantity,Apr 2011 - Installations Quantity,May 2011 - Installations Quantity,Jun 2011 - Installations Quantity,Jul 2011 - Installations Quantity,Aug 2011 - Installations Quantity,Sep 2011 - Installations Quantity,...,Jul 2023 - Installations Quantity,Aug 2023 - Installations Quantity,Sep 2023 - Installations Quantity,Oct 2023 - Installations Quantity,Nov 2023 - Installations Quantity,Dec 2023 - Installations Quantity,Jan 2024 - Installations Quantity,Feb 2024 - Installations Quantity,Mar 2024 - Installations Quantity,Apr 2024 - Installations Quantity
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,200,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,800,0,0,0,0,1,0,1,0,0,...,1,2,0,0,1,0,0,0,0,0
3,801,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,803,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [37]:
df_installations['State'] = df_installations['Small Unit Installation Postcode'].apply(get_state_from_postcode)
state_col = df_installations.pop('State')
df_installations.insert(1, 'State', state_col)
df_installations_melbourne = df_installations[df_installations['State'] == 'VIC']
df_installations_melbourne.head()

C:\Users\aynss\AppData\Local\Temp\ipykernel_26568\3574573191.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_installations.insert(1, 'State', state_col)


,Small Unit Installation Postcode,State,Jan 2011 - Installations Quantity,Feb 2011 - Installations Quantity,Mar 2011 - Installations Quantity,Apr 2011 - Installations Quantity,May 2011 - Installations Quantity,Jun 2011 - Installations Quantity,Jul 2011 - Installations Quantity,Aug 2011 - Installations Quantity,...,Jul 2023 - Installations Quantity,Aug 2023 - Installations Quantity,Sep 2023 - Installations Quantity,Oct 2023 - Installations Quantity,Nov 2023 - Installations Quantity,Dec 2023 - Installations Quantity,Jan 2024 - Installations Quantity,Feb 2024 - Installations Quantity,Mar 2024 - Installations Quantity,Apr 2024 - Installations Quantity
736,3000,VIC,0,0,0,0,1,2,0,0,...,2,1,1,1,1,3,0,0,0,0
737,3002,VIC,0,0,0,0,1,1,1,0,...,1,2,0,0,1,0,0,1,0,0
738,3003,VIC,0,0,1,0,1,1,0,0,...,2,3,0,0,1,1,0,0,0,0
739,3004,VIC,0,0,0,0,0,0,0,0,...,0,0,0,0,1,1,0,0,0,0
740,3006,VIC,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,0


In [36]:
# df_installation_long = df_installations_melbourne.melt(
#     id_vars['Small Unit Installation Postcode'],
#     var_name='Raw_Date', 
#     value_name='Installation_Quantity'
# )
# df_capacity_long['Date'] = pd.to_datetime(
#     df_capacity_long['Raw_Date'].str.extract(r'([Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec]+ \d{4})')[0],
#     format='%b %Y'
# )
# df_capacity_long = df_capacity_long.drop(columns=['Raw_Date'])
# df_capacity_long.head(20)

NameError: name 'id_vars' is not defined

In [39]:
install_cols = df_installations_melbourne.columns[2:]

# 特徴量③: 累計の太陽光設置件数
df_installations_melbourne['Total_Installations'] = df_installations_melbourne[install_cols].sum(axis=1)

df_inst_features = df_installations_melbourne[['Small Unit Installation Postcode', 'Total_Installations']].copy()
df_inst_features.rename(columns={'Small Unit Installation Postcode': 'Postcode'}, inplace=True)
df_inst_features.head()

,Postcode,Total_Installations
736,3000,178
737,3002,176
738,3003,234
739,3004,38
740,3006,78


In [6]:
# Melbourne house
df_mel_housing = pd.read_csv('MELBOURNE_HOUSE_PRICES_LESS.csv')
df_mel_housing.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Postcode,Regionname,Propertycount,Distance,CouncilArea
0,Abbotsford,49 Lithgow St,3,h,1490000.0,S,Jellis,1/04/2017,3067,Northern Metropolitan,4019,3.0,Yarra City Council
1,Abbotsford,59A Turner St,3,h,1220000.0,S,Marshall,1/04/2017,3067,Northern Metropolitan,4019,3.0,Yarra City Council
2,Abbotsford,119B Yarra St,3,h,1420000.0,S,Nelson,1/04/2017,3067,Northern Metropolitan,4019,3.0,Yarra City Council
3,Aberfeldie,68 Vida St,3,h,1515000.0,S,Barry,1/04/2017,3040,Western Metropolitan,1543,7.5,Moonee Valley City Council
4,Airport West,92 Clydesdale Rd,2,h,670000.0,S,Nelson,1/04/2017,3042,Western Metropolitan,3464,10.4,Moonee Valley City Council


In [40]:
# ==========================================
# 3. メルボルン住宅価格データから特徴量を抽出（Postcode単位に集約）
# ==========================================
# 特徴量④⑤⑥: 平均価格、平均部屋数、総物件数、中心地からの平均距離
df_housing_features = df_mel_housing.groupby('Postcode').agg({
    'Price': 'median',       # 価格は外れ値に強い中央値がおすすめ
    'Rooms': 'mean',         # 平均部屋数（家の大きさのプロキシ）
    'Distance': 'mean',      # メルボルン中心地からの距離
    'Propertycount': 'max'   # そのエリアの総物件数（世帯数のプロキシ）
}).reset_index()

In [41]:
# ==========================================
# 4. 地理データ（緯度・経度）の抽出
# ==========================================
# 重複を排除して、Postcodeごとの代表的な緯度・経度を取得
df_geo = df_vic_postcode.groupby('postcode').agg({
    'long': 'mean',
    'lat': 'mean',
    'locality': 'first'  # 代表的な地域名
}).reset_index()
df_geo.rename(columns={'postcode': 'Postcode'}, inplace=True)

In [42]:
# ==========================================
# 5. すべてのデータをマージ（結合）
# ==========================================
# 住宅データをベースに、太陽光データと地理データを結合
analysis_df = df_housing_features.merge(df_cap_features, on='Postcode', how='left')
analysis_df = analysis_df.merge(df_inst_features, on='Postcode', how='left')
analysis_df = analysis_df.merge(df_geo, on='Postcode', how='left')

# 太陽光データがなくてNaNになった場所は0で埋める
analysis_df['Total_Capacity_kW'] = analysis_df['Total_Capacity_kW'].fillna(0)
analysis_df['Recent_Capacity_kW'] = analysis_df['Recent_Capacity_kW'].fillna(0)
analysis_df['Total_Installations'] = analysis_df['Total_Installations'].fillna(0)

In [43]:
# ==========================================
# 6. 高度な特徴量抽出 (Advanced Feature Extraction)
# ==========================================
# 要件である「特徴量抽出のハイライト」として、単純な集計ではない指標を作ります。

# 特徴量⑦: 太陽光普及率（設置数 ÷ 総物件数）
analysis_df['Solar_Adoption_Rate'] = analysis_df['Total_Installations'] / analysis_df['Propertycount']

# 特徴量⑧: 1件あたりの平均発電容量（大型のシステムが好まれるエリアか）
analysis_df['Avg_Capacity_per_Install'] = analysis_df['Total_Capacity_kW'] / analysis_df['Total_Installations']
analysis_df['Avg_Capacity_per_Install'] = analysis_df['Avg_Capacity_per_Install'].fillna(0)

# 最後に、価格や距離などのデータが欠損している行をドロップ
analysis_df = analysis_df.dropna(subset=['Price', 'Distance']).reset_index(drop=True)

print(f"分析用データセットが完成しました！ 行数: {analysis_df.shape[0]}, 列数: {analysis_df.shape[1]}")
analysis_df.head()

分析用データセットが完成しました！ 行数: 221, 列数: 13


,Postcode,Price,Rooms,Distance,Propertycount,Total_Capacity_kW,Recent_Capacity_kW,Total_Installations,long,lat,locality,Solar_Adoption_Rate,Avg_Capacity_per_Install
0,3000,648000.0,1.972125,0.0,17496,5158.312,716.670,178,144.982585,-37.814437,MELBOURNE,0.010174,28.979281
1,3002,1000000.0,2.277108,1.3,3040,1521.170,82.910,176,144.982585,-37.814437,EAST MELBOURNE,0.057895,8.643011
2,3003,890000.0,2.500000,3.1,2230,3659.060,285.975,234,144.949592,-37.810871,WEST MELBOURNE,0.104933,15.637009
3,3006,595000.0,2.000000,0.7,8400,2643.510,176.840,78,144.959000,-37.824272,SOUTH WHARF,0.009286,33.891154
4,3008,625000.0,2.150000,1.4,4707,3585.850,276.195,220,144.948039,-37.814719,DOCKLANDS,0.046739,16.299318


In [ ]:
%pip install matplotlib
%pip install seaborn
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. クラスタリング用特徴量の選定と標準化
# ==========================================
# 分析に使用する特徴量を厳選（スケーリングが必須です）
cluster_features = [
    "Price",
    "Distance",
    "Solar_Adoption_Rate",
    "Avg_Capacity_per_Install",
]
X = analysis_df[cluster_features].copy()

# 平均0、分散1に標準化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ==========================================
# 2. エルボー法とシルエットスコアによる最適なKの探索
# ==========================================
inertia = []
silhouette_avg = []
K_range = range(2, 7)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)

    inertia.append(kmeans.inertia_)
    silhouette_avg.append(silhouette_score(X_scaled, cluster_labels))

# 評価グラフの描画
fig, ax1 = plt.subplots(figsize=(10, 4))

# エルボー係数（慣性）のプロット
color = "tab:blue"
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("Inertia (Elbow Method)", color=color)
ax1.plot(K_range, inertia, "o-", color=color)
ax1.tick_params(axis="y", labelcolor=color)

# シルエットスコアのプロット
ax2 = ax1.twinx()
color = "tab:red"
ax2.set_ylabel("Silhouette Score", color=color)
ax2.plot(K_range, silhouette_avg, "s--", color=color)
ax2.tick_params(axis="y", labelcolor=color)

plt.title("Evaluation of Clustering (Elbow vs Silhouette)")
fig.tight_layout()
plt.show()

# ==========================================
# 3. 最適なクラスタ数（例: k=3）でファイナライズ
# ==========================================
# グラフの結果、エルボーの曲がり角でありシルエットスコアが比較的高い「k=3」を仮定
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
analysis_df["Cluster"] = kmeans.fit_predict(X_scaled)

print(
    f"クラスタリング完了。シルエットスコア: {silhouette_score(X_scaled, analysis_df['Cluster']):.3f}"
)

In [ ]:
from scipy import stats

# クラスタごとの太陽光普及率データを抽出
groups = [
    group["Solar_Adoption_Rate"].values
    for name, group in analysis_df.groupby("Cluster")
]

# Kruskal-Wallis検定の実行
stat, p_value = stats.kruskal(*groups)

print("--- 有意差検定（Significance Testing）の結果 ---")
print(f"Kruskal-Wallis H-statistic: {stat:.4f}")
print(f"p-value: {p_value:.5e}")

if p_value < 0.05:
    print(
        "\n【結論】p値が0.05未満であるため、クラスタ間で太陽光普及率に『統計的に有意な差』が認められます。"
    )
    print(
        "これは、各クラスタが異なる背景（市場特性）を持っており、一律の政策ではなくクラスタごとに最適化したアプローチ（政策提言）が有効であることを示唆しています。"
    )
else:
    print(
        "\n【結論】統計的な有意差は見られませんでした。クラスタ数の再検討が必要です。"
    )

In [ ]:
# 各クラスタの平均値を算出して特徴を掴む
cluster_summary = (
    analysis_df.groupby("Cluster")[
        ["Price", "Distance", "Solar_Adoption_Rate", "Avg_Capacity_per_Install"]
    ]
    .mean()
    .reset_index()
)
print("\n--- 各クラスタの特徴一覧 ---")
print(cluster_summary)

**Specific requirements for this project**

 

The following analysis and techniques must be demonstrated as part of the above to fully meet the requirements of the skillset/cluster: 

Discussion of at least two clustering techniques and their appropriateness in this context 

Application of at least one clustering technique that is appropriate in the context 

You should include techniques to analyse the effectiveness of clustering and interpretation of the clusters 

Use of significance testing on at least one occasion 

Demonstrate and highlight usage of feature extraction 

Appropriate visualization of the results (geospatial visualization is recommended)

avarage house how looks like? bedrooms/price/landsize
solor adoption from postcode: install/capacity